### Dataset Visualization

#### Visualization the Exact Datset Frames

In [31]:
# from PIL import ImageFont, ImageDraw, Image
# import matplotlib.pyplot as plt
# import pandas as pd
# import numpy as np
# import os

# # ============================================================
# # CONFIGURATION
# # ============================================================
# VIDEO_ID = "1.1.1"
# FRAME_DIR = f"/home1/jtt_1/mtp/okutama-action/TrainSetFrames/Drone1/Morning/Extracted-Frames-1280x720/{VIDEO_ID}"
# LABEL_FILE = f"/home1/jtt_1/mtp/okutama-action/TrainSetFrames/Labels/MultiActionLabels/3840x2160/{VIDEO_ID}.txt"
# NUM_FRAMES_TO_SHOW = 1

# FRAME_WIDTH, FRAME_HEIGHT = 1280, 720
# LABEL_WIDTH, LABEL_HEIGHT = 3840, 2160

# scale_x = FRAME_WIDTH / LABEL_WIDTH
# scale_y = FRAME_HEIGHT / LABEL_HEIGHT

# # ✅ Choose your font size
# FONT_SIZE = 18  # try 16–24 for 720p frames

# # Try loading a proper font
# try:
#     font = ImageFont.truetype("DejaVuSans-Bold.ttf", FONT_SIZE)
# except:
#     font = ImageFont.load_default()

# # ============================================================
# # LOAD LABEL FILE FLEXIBLY
# # ============================================================
# with open(LABEL_FILE, "r") as f:
#     lines = [l.strip().split() for l in f if l.strip()]
# max_len = max(len(x) for x in lines)
# padded_lines = [x + [None]*(max_len - len(x)) for x in lines]

# base_cols = [
#     "track_id", "xmin", "ymin", "xmax", "ymax",
#     "frame", "lost", "occluded", "generated", "label"
# ]
# extra_cols = [f"class_{i}" for i in range(max_len - len(base_cols))]
# all_cols = base_cols + extra_cols

# df = pd.DataFrame(padded_lines, columns=all_cols).apply(pd.to_numeric, errors="ignore")

# # ============================================================
# # CLEAN LABELS
# # ============================================================
# def clean_action(x):
#     if pd.isna(x): return None
#     x = str(x).strip().replace('"', '').replace("'", "")
#     if x in ["Hand", "Shaking", "HandShaking"]: return "Hand Shaking"
#     if x == "Pushing/Pulling": return "Pushing or Pulling"
#     return x

# for c in extra_cols:
#     df[c] = df[c].apply(clean_action)

# # ============================================================
# # VISUALIZATION
# # ============================================================
# frames_to_show = sorted(df["frame"].dropna().unique())[:NUM_FRAMES_TO_SHOW]
# frame_files = [f for f in os.listdir(FRAME_DIR) if f.endswith(".jpg")]
# use_simple_numbers = any(f == "0.jpg" for f in frame_files)

# for frame_num in frames_to_show:
#     frame_path = os.path.join(FRAME_DIR, f"{int(frame_num)}.jpg" if use_simple_numbers else f"frame_{int(frame_num):06d}.jpg")
#     if not os.path.exists(frame_path):
#         print(f"⚠️ Missing frame: {frame_path}")
#         continue

#     img = Image.open(frame_path).convert("RGB")
#     draw = ImageDraw.Draw(img)
#     frame_df = df[df["frame"] == frame_num]

#     for _, row in frame_df.iterrows():
#         if row["lost"] == 1 or row["occluded"] == 1:
#             continue

#         # Scale bounding boxes
#         xmin = float(row["xmin"]) * scale_x
#         ymin = float(row["ymin"]) * scale_y
#         xmax = float(row["xmax"]) * scale_x
#         ymax = float(row["ymax"]) * scale_y

#         draw.rectangle([xmin, ymin, xmax, ymax], outline="lime", width=2)

#         # Multi-label text
#         actions = [a for a in extra_cols if row[a] not in [None, np.nan]]
#         active_actions = [row[a] for a in actions if pd.notna(row[a])]
#         text = f"ID {int(row['track_id'])}: {', '.join(active_actions)}" if active_actions else f"ID {int(row['track_id'])}"

#         # Compute text size using new Pillow API
#         try:
#             # For Pillow >=10
#             text_bbox = draw.textbbox((0, 0), text, font=font)
#             text_w = text_bbox[2] - text_bbox[0]
#             text_h = text_bbox[3] - text_bbox[1]
#         except AttributeError:
#             # For older Pillow
#             text_w, text_h = font.getsize(text)

#         text_x, text_y = xmin, max(ymin - text_h - 4, 0)

#         # Black background for text
#         draw.rectangle(
#             [text_x, text_y, text_x + text_w + 6, text_y + text_h + 4],
#             fill="black"
#         )

#         # Green text on top
#         draw.text((text_x + 3, text_y + 2), text, fill="lime", font=font)

#     plt.figure(figsize=(8, 6))
#     plt.imshow(img)
#     plt.title(f"Video {VIDEO_ID} | Frame {int(frame_num)} (Font size: {FONT_SIZE})")
#     plt.axis("off")
#     plt.show()


#### Convert One Vedio

In [32]:
# from PIL import ImageFont, ImageDraw, Image
# import pandas as pd
# import numpy as np
# import cv2
# import os
# from tqdm import tqdm

# # ============================================================
# # CONFIGURATION
# # ============================================================
# VIDEO_ID = "1.1.3"

# FRAME_DIR = f"/home1/jtt_1/mtp/okutama-action/TrainSetFrames/Drone1/Morning/Extracted-Frames-1280x720/{VIDEO_ID}"
# LABEL_FILE = f"/home1/jtt_1/mtp/okutama-action/TrainSetFrames/Labels/MultiActionLabels/3840x2160/{VIDEO_ID}.txt"
# OUTPUT_PATH = f"AnnotatedVideos/Morning/{VIDEO_ID}_annotated.mp4"

# os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# FRAME_WIDTH, FRAME_HEIGHT = 1280, 720
# LABEL_WIDTH, LABEL_HEIGHT = 3840, 2160
# scale_x = FRAME_WIDTH / LABEL_WIDTH
# scale_y = FRAME_HEIGHT / LABEL_HEIGHT
# FPS = 15

# # ============================================================
# # FONT SETUP
# # ============================================================
# FONT_SIZE = 18
# try:
#     font = ImageFont.truetype("DejaVuSans-Bold.ttf", FONT_SIZE)
# except:
#     font = ImageFont.load_default()

# # ============================================================
# # HELPER FUNCTIONS
# # ============================================================

# def clean_action(x):
#     """Normalize label names"""
#     if pd.isna(x): return None
#     x = str(x).strip().replace('"', '').replace("'", "")
#     if x in ["Hand", "Shaking", "HandShaking"]: return "Hand Shaking"
#     if x == "Pushing/Pulling": return "Pushing or Pulling"
#     return x

# def get_color(track_id, track_colors):
#     """Assign unique color to each track"""
#     if track_id not in track_colors:
#         track_colors[track_id] = tuple(np.random.randint(0, 255, 3).tolist())
#     return track_colors[track_id]

# # ============================================================
# # LOAD LABEL FILE
# # ============================================================
# if not os.path.exists(LABEL_FILE):
#     raise FileNotFoundError(f"❌ Label file not found: {LABEL_FILE}")

# with open(LABEL_FILE, "r") as f:
#     lines = [l.strip().split() for l in f if l.strip()]

# max_len = max(len(x) for x in lines)
# padded_lines = [x + [None] * (max_len - len(x)) for x in lines]

# base_cols = [
#     "track_id", "xmin", "ymin", "xmax", "ymax",
#     "frame", "lost", "occluded", "generated", "label"
# ]
# extra_cols = [f"class_{i}" for i in range(max_len - len(base_cols))]
# all_cols = base_cols + extra_cols

# df = pd.DataFrame(padded_lines, columns=all_cols).apply(pd.to_numeric, errors="ignore")
# for c in extra_cols:
#     df[c] = df[c].apply(clean_action)

# # Pre-group frames for speed
# frame_groups = dict(tuple(df.groupby("frame")))

# # ============================================================
# # PREPARE FRAME LIST
# # ============================================================
# frame_files = sorted(
#     [f for f in os.listdir(FRAME_DIR) if f.endswith(".jpg")],
#     key=lambda x: int(os.path.splitext(x)[0])
# )
# if not frame_files:
#     raise RuntimeError(f"⚠️ No frames found in {FRAME_DIR}")

# print(f"🖼 Found {len(frame_files)} frames for video {VIDEO_ID}")

# # ============================================================
# # SETUP VIDEO WRITER
# # ============================================================
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out = cv2.VideoWriter(OUTPUT_PATH, fourcc, FPS, (FRAME_WIDTH, FRAME_HEIGHT))

# track_colors = {}

# # ============================================================
# # LOOP OVER FRAMES
# # ============================================================
# for frame_name in tqdm(frame_files, desc=f"Annotating video {VIDEO_ID}"):
#     frame_num = int(os.path.splitext(frame_name)[0])
#     frame_path = os.path.join(FRAME_DIR, frame_name)

#     if not os.path.exists(frame_path):
#         print(f"⚠️ Missing frame: {frame_path}")
#         continue

#     img = Image.open(frame_path).convert("RGB")
#     draw = ImageDraw.Draw(img)
#     frame_df = frame_groups.get(frame_num, pd.DataFrame())

#     for _, row in frame_df.iterrows():
#         if row["lost"] == 1 or row["occluded"] == 1:
#             continue

#         # Scale bounding boxes
#         xmin = float(row["xmin"]) * scale_x
#         ymin = float(row["ymin"]) * scale_y
#         xmax = float(row["xmax"]) * scale_x
#         ymax = float(row["ymax"]) * scale_y

#         color = get_color(row["track_id"], track_colors)
#         draw.rectangle([xmin, ymin, xmax, ymax], outline=color, width=2)

#         # Multi-label actions
#         active_actions = [row[a] for a in extra_cols if pd.notna(row[a])]
#         text = f"ID {int(row['track_id'])}: {', '.join(active_actions)}" if active_actions else f"ID {int(row['track_id'])}"

#         try:
#             text_bbox = draw.textbbox((0, 0), text, font=font)
#             text_w = text_bbox[2] - text_bbox[0]
#             text_h = text_bbox[3] - text_bbox[1]
#         except AttributeError:
#             text_w, text_h = font.getsize(text)

#         text_x, text_y = xmin, max(ymin - text_h - 4, 0)
#         draw.rectangle([text_x, text_y, text_x + text_w + 6, text_y + text_h + 4], fill="black")
#         draw.text((text_x + 3, text_y + 2), text, fill="lime", font=font)

#     # Optional frame overlay info
#     draw.text((10, 10), f"Frame: {frame_num}", fill="white", font=font)

#     frame_out = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
#     out.write(frame_out)

# out.release()
# print(f"\n✅ Annotated video saved successfully at:\n{OUTPUT_PATH}")


### Dataset Creation Scene-level and Mix

##### Train Data Frames

In [33]:
# import os
# import cv2
# import pandas as pd
# import numpy as np
# from tqdm import tqdm

# # ============================================================
# # CONFIGURATION
# # ============================================================
# ROOT = "/home1/jtt_1/mtp/okutama-action/TrainSetFrames"
# LABEL_DIR = os.path.join(ROOT, "Labels/MultiActionLabels/3840x2160")
# OUTPUT_ROOT = "timesformer-dataset-train"

# FRAME_WIDTH, FRAME_HEIGHT = 1280, 720
# LABEL_WIDTH, LABEL_HEIGHT = 3840, 2160
# FPS = 10

# scale_x = FRAME_WIDTH / LABEL_WIDTH
# scale_y = FRAME_HEIGHT / LABEL_HEIGHT

# os.makedirs(OUTPUT_ROOT, exist_ok=True)

# # ============================================================
# # HELPER FUNCTIONS
# # ============================================================
# def clean_action(x):
#     """Normalize action names to consistent labels."""
#     if pd.isna(x):
#         return None
#     x = str(x).strip().replace('"', '').replace("'", "")
#     if x in ["Hand", "Shaking", "HandShaking"]:
#         return "Handshaking"
#     if x == "Pushing/Pulling":
#         return "Pushing or Pulling"
#     return x


# def load_labels(label_file):
#     """Load bounding box labels for each frame."""
#     with open(label_file, "r") as f:
#         lines = [l.strip().split() for l in f if l.strip()]
#     max_len = max(len(x) for x in lines)
#     padded = [x + [None] * (max_len - len(x)) for x in lines]

#     base_cols = [
#         "track_id", "xmin", "ymin", "xmax", "ymax",
#         "frame", "lost", "occluded", "generated", "label"
#     ]
#     extra_cols = [f"class_{i}" for i in range(max_len - len(base_cols))]
#     cols = base_cols + extra_cols

#     df = pd.DataFrame(padded, columns=cols)
#     for c in extra_cols:
#         df[c] = df[c].apply(clean_action)
#     df = df.apply(pd.to_numeric, errors="ignore")
#     return df, extra_cols


# # ============================================================
# # MAIN CONVERSION
# # ============================================================
# drones = ["Drone1", "Drone2"]
# times_of_day = ["Morning", "Noon"]

# summary_rows = []

# print("🚀 Starting TimeSformer dataset creation (clean videos + bbox CSVs)...\n")

# for drone in drones:
#     for time in times_of_day:
#         frame_base = os.path.join(ROOT, drone, time, "Extracted-Frames-1280x720")
#         if not os.path.exists(frame_base):
#             print(f"⚠️ Missing folder: {frame_base}")
#             continue

#         output_dir = os.path.join(OUTPUT_ROOT, drone, time)
#         os.makedirs(output_dir, exist_ok=True)

#         video_folders = sorted([
#             d for d in os.listdir(frame_base)
#             if os.path.isdir(os.path.join(frame_base, d))
#         ])
#         print(f"🎥 {drone}/{time} → {len(video_folders)} videos to process")

#         for vid in tqdm(video_folders, desc=f"{drone}-{time}"):
#             frame_dir = os.path.join(frame_base, vid)
#             label_file = os.path.join(LABEL_DIR, f"{vid}.txt")
#             output_video = os.path.join(output_dir, f"{vid}.mp4")
#             bbox_csv_path = os.path.join(output_dir, f"{vid}_bboxes.csv")

#             # 🔹 Skip missing label files
#             if not os.path.exists(label_file):
#                 print(f"⚠️ No labels for {vid}")
#                 continue

#             # 🔹 Skip if already processed
#             if os.path.exists(output_video) and os.path.exists(bbox_csv_path):
#                 print(f"⏩ Skipping {vid}, already processed.")
#                 continue

#             # Load labels
#             df, action_cols = load_labels(label_file)

#             # Scale bbox coordinates
#             for col in ["xmin", "xmax"]:
#                 df[col] = df[col].astype(float) * scale_x
#             for col in ["ymin", "ymax"]:
#                 df[col] = df[col].astype(float) * scale_y

#             # Combine actions into single string
#             df["actions"] = df[action_cols].apply(
#                 lambda r: ",".join([str(a) for a in r if pd.notna(a)]), axis=1
#             )

#             # Save frame-level bbox CSV
#             df_out = df[[
#                 "frame", "track_id", "xmin", "ymin", "xmax", "ymax",
#                 "lost", "occluded", "actions"
#             ]]
#             df_out.to_csv(bbox_csv_path, index=False)

#             # Gather unique actions
#             all_actions = sorted(set([
#                 a for a in df[action_cols].values.ravel() if pd.notna(a)
#             ]))
#             all_actions_str = ",".join(all_actions)

#             # Gather frame list
#             frame_files = sorted([
#                 f for f in os.listdir(frame_dir) if f.endswith(".jpg")
#             ], key=lambda x: int(os.path.splitext(x)[0]) if os.path.splitext(x)[0].isdigit() else -1)

#             # 🔹 Skip if no frames found
#             if not frame_files:
#                 print(f"⚠️ No frames found in {frame_dir}")
#                 continue

#             # Validate frame naming consistency
#             if not all(f.split('.')[0].isdigit() for f in frame_files):
#                 print(f"⚠️ Non-numeric frame names detected in {vid}. Skipping.")
#                 continue

#             # Initialize video writer
#             first_frame = cv2.imread(os.path.join(frame_dir, frame_files[0]))
#             if first_frame is None:
#                 print(f"⚠️ Could not read first frame of {vid}. Skipping.")
#                 continue

#             height, width, _ = first_frame.shape
#             writer = cv2.VideoWriter(
#                 output_video,
#                 cv2.VideoWriter_fourcc(*"mp4v"),
#                 FPS,
#                 (width, height)
#             )

#             # Write clean frames
#             for f in frame_files:
#                 frame_path = os.path.join(frame_dir, f)
#                 img = cv2.imread(frame_path)
#                 if img is None:
#                     print(f"⚠️ Skipping unreadable frame: {frame_path}")
#                     continue
#                 writer.write(img)

#             writer.release()

#             summary_rows.append([
#                 drone, time, vid, len(frame_files),
#                 all_actions_str, FPS, output_video, bbox_csv_path
#             ])

# # ============================================================
# # SAVE SUMMARY CSV
# # ============================================================
# summary_df = pd.DataFrame(summary_rows, columns=[
#     "Drone", "TimeOfDay", "VideoID", "FrameCount",
#     "Actions", "FPS", "VideoPath", "BBoxCSVPath"
# ])

# csv_path = os.path.join(OUTPUT_ROOT, "labels.csv")
# summary_df.to_csv(csv_path, index=False)

# print(f"\n✅ Dataset ready for TimeSformer (clean videos + bbox CSVs).")
# print(f"📁 Videos & bbox CSVs saved to: {OUTPUT_ROOT}")
# print(f"🧾 Summary CSV saved to: {csv_path}")


##### Test Data Frames

In [34]:
# import os
# import cv2
# import pandas as pd
# import numpy as np
# from tqdm import tqdm

# # ============================================================
# # CONFIGURATION
# # ============================================================
# ROOT = "/home1/jtt_1/mtp/okutama-action/TestSetFrames"
# LABEL_DIR = os.path.join(ROOT, "Labels/MultiActionLabels/3840x2160")
# OUTPUT_ROOT = "timesformer-dataset-test"

# FRAME_WIDTH, FRAME_HEIGHT = 1280, 720
# LABEL_WIDTH, LABEL_HEIGHT = 3840, 2160
# FPS = 20

# scale_x = FRAME_WIDTH / LABEL_WIDTH
# scale_y = FRAME_HEIGHT / LABEL_HEIGHT

# os.makedirs(OUTPUT_ROOT, exist_ok=True)

# # ============================================================
# # HELPER FUNCTIONS
# # ============================================================
# def clean_action(x):
#     """Normalize action names to consistent labels."""
#     if pd.isna(x):
#         return None
#     x = str(x).strip().replace('"', '').replace("'", "")
#     if x in ["Hand", "Shaking", "HandShaking"]:
#         return "Handshaking"
#     if x == "Pushing/Pulling":
#         return "Pushing or Pulling"
#     return x


# def load_labels(label_file):
#     """Load bounding box labels for each frame."""
#     with open(label_file, "r") as f:
#         lines = [l.strip().split() for l in f if l.strip()]
#     max_len = max(len(x) for x in lines)
#     padded = [x + [None] * (max_len - len(x)) for x in lines]

#     base_cols = ["track_id", "xmin", "ymin", "xmax", "ymax",
#                  "frame", "lost", "occluded", "generated", "label"]
#     extra_cols = [f"class_{i}" for i in range(max_len - len(base_cols))]
#     cols = base_cols + extra_cols

#     df = pd.DataFrame(padded, columns=cols)
#     for c in extra_cols:
#         df[c] = df[c].apply(clean_action)
#     df = df.apply(pd.to_numeric, errors="ignore")
#     return df, extra_cols


# # ============================================================
# # MAIN CONVERSION
# # ============================================================
# drones = ["Drone1", "Drone2"]
# times_of_day = ["Morning", "Noon"]

# summary_rows = []

# print("🚀 Starting TimeSformer TEST dataset creation (clean videos + bbox CSVs)...\n")

# for drone in drones:
#     for time in times_of_day:
#         frame_base = os.path.join(ROOT, drone, time, "Extracted-Frames-1280x720")
#         if not os.path.exists(frame_base):
#             print(f"⚠️ Missing folder: {frame_base}")
#             continue

#         output_dir = os.path.join(OUTPUT_ROOT, drone, time)
#         os.makedirs(output_dir, exist_ok=True)

#         video_folders = sorted([
#             d for d in os.listdir(frame_base)
#             if os.path.isdir(os.path.join(frame_base, d))
#         ])
#         print(f"🎥 {drone}/{time} → {len(video_folders)} videos to process")

#         for vid in tqdm(video_folders, desc=f"{drone}-{time}"):
#             frame_dir = os.path.join(frame_base, vid)
#             label_file = os.path.join(LABEL_DIR, f"{vid}.txt")
#             output_video = os.path.join(output_dir, f"{vid}.mp4")
#             bbox_csv_path = os.path.join(output_dir, f"{vid}_bboxes.csv")

#             if not os.path.exists(label_file):
#                 print(f"⚠️ No labels for {vid}")
#                 continue

#             # Skip if already processed
#             if os.path.exists(output_video) and os.path.exists(bbox_csv_path):
#                 print(f"⏩ Skipping {vid}, already processed.")
#                 continue

#             # Load labels
#             df, action_cols = load_labels(label_file)

#             # Scale bbox coordinates
#             for col in ["xmin", "xmax"]:
#                 df[col] = df[col].astype(float) * scale_x
#             for col in ["ymin", "ymax"]:
#                 df[col] = df[col].astype(float) * scale_y

#             # Combine multi-actions
#             df["actions"] = df[action_cols].apply(
#                 lambda r: ",".join([str(a) for a in r if pd.notna(a)]), axis=1
#             )

#             # Save bbox CSV
#             df_out = df[[
#                 "frame", "track_id", "xmin", "ymin", "xmax", "ymax",
#                 "lost", "occluded", "actions"
#             ]]
#             df_out.to_csv(bbox_csv_path, index=False)

#             # Collect all unique actions
#             all_actions = sorted(set([
#                 a for a in df[action_cols].values.ravel() if pd.notna(a)
#             ]))
#             all_actions_str = ",".join(all_actions)

#             # Collect frames
#             frame_files = sorted([
#                 f for f in os.listdir(frame_dir) if f.endswith(".jpg")
#             ], key=lambda x: int(os.path.splitext(x)[0]) if os.path.splitext(x)[0].isdigit() else -1)

#             if not frame_files:
#                 print(f"⚠️ No frames found in {frame_dir}")
#                 continue

#             # Validate frame names
#             if not all(f.split('.')[0].isdigit() for f in frame_files):
#                 print(f"⚠️ Non-numeric frame names in {vid}, skipping.")
#                 continue

#             # Initialize video writer
#             first_frame = cv2.imread(os.path.join(frame_dir, frame_files[0]))
#             if first_frame is None:
#                 print(f"⚠️ Cannot read first frame of {vid}, skipping.")
#                 continue

#             height, width, _ = first_frame.shape
#             writer = cv2.VideoWriter(
#                 output_video,
#                 cv2.VideoWriter_fourcc(*"mp4v"),
#                 FPS,
#                 (width, height)
#             )

#             # Write clean video frames
#             for f in frame_files:
#                 frame_path = os.path.join(frame_dir, f)
#                 img = cv2.imread(frame_path)
#                 if img is None:
#                     print(f"⚠️ Unreadable frame: {frame_path}")
#                     continue
#                 writer.write(img)

#             writer.release()

#             summary_rows.append([
#                 drone, time, vid, len(frame_files),
#                 all_actions_str, FPS, output_video, bbox_csv_path
#             ])

# # ============================================================
# # SAVE SUMMARY CSV
# # ============================================================
# summary_df = pd.DataFrame(summary_rows, columns=[
#     "Drone", "TimeOfDay", "VideoID", "FrameCount",
#     "Actions", "FPS", "VideoPath", "BBoxCSVPath"
# ])

# csv_path = os.path.join(OUTPUT_ROOT, "labels.csv")
# summary_df.to_csv(csv_path, index=False)

# print(f"\n✅ Test dataset ready for TimeSformer (clean videos + bbox CSVs).")
# print(f"📁 Videos & bbox CSVs saved to: {OUTPUT_ROOT}")
# print(f"🧾 Summary CSV saved to: {csv_path}")


#### Visualization of converted Dataset videos

In [35]:
# import os
# import cv2
# import pandas as pd
# import numpy as np
# from tqdm import tqdm
# import random

# # ============================================================
# # CONFIGURATION
# # ============================================================
# CSV_PATH = "/home1/jtt_1/mtp/timesformer-dataset-test/labels.csv"
# OUTPUT_DIR = "/home1/jtt_1/mtp/timesformer-visualizations-test"
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# MAX_FRAMES = 500
# NUM_VIDEOS = 4
# DRAW_TEXT = True
# DRAW_TRACK_ID = True

# # Random color per track ID
# track_colors = {}
# def get_color(track_id):
#     if track_id not in track_colors:
#         track_colors[track_id] = tuple(np.random.randint(0, 255, 3).tolist())
#     return track_colors[track_id]

# # ============================================================
# # LOAD CSV
# # ============================================================
# if not os.path.exists(CSV_PATH):
#     raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

# df = pd.read_csv(CSV_PATH)
# print(f"📄 Loaded {len(df)} video entries from {CSV_PATH}")

# df_sample = df.sample(n=NUM_VIDEOS, random_state=None).reset_index(drop=True)
# print(f"\n🎲 Randomly selected {NUM_VIDEOS} videos for visualization:")

# # ============================================================
# # VISUALIZATION LOOP
# # ============================================================
# for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Visualizing videos"):
#     video_path = str(row["VideoPath"]).strip()
#     bbox_csv = str(row["BBoxCSVPath"]).strip()
#     video_id = str(row["VideoID"])

#     if not os.path.exists(video_path):
#         print(f"⚠️ Missing video: {video_path}")
#         continue
#     if not os.path.exists(bbox_csv):
#         print(f"⚠️ Missing bbox CSV: {bbox_csv}")
#         continue

#     df_bboxes = pd.read_csv(bbox_csv)
#     df_bboxes["frame"] = df_bboxes["frame"].astype(int)
#     min_frame = df_bboxes["frame"].min()

#     cap = cv2.VideoCapture(video_path)
#     if not cap.isOpened():
#         print(f"⚠️ Cannot open video {video_path}")
#         continue

#     fps = cap.get(cv2.CAP_PROP_FPS)
#     width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#     total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

#     out_path = os.path.join(OUTPUT_DIR, f"{video_id}_bbox_preview.mp4")
#     writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

#     print(f"\n🎥 {video_id}: {min(MAX_FRAMES, total_frames)} frames → {out_path}")

#     frame_idx = 0
#     while cap.isOpened() and frame_idx < MAX_FRAMES:
#         ret, frame = cap.read()
#         if not ret:
#             break

#         true_frame_idx = frame_idx + min_frame  # offset for 1-based labels
#         frame_df = df_bboxes[df_bboxes["frame"] == true_frame_idx]

#         for _, bbox_row in frame_df.iterrows():
#             if int(bbox_row["lost"]) == 1 or int(bbox_row["occluded"]) == 1:
#                 continue

#             tid = int(bbox_row["track_id"])
#             color = get_color(tid)
#             x1, y1, x2, y2 = map(int, [bbox_row["xmin"], bbox_row["ymin"], bbox_row["xmax"], bbox_row["ymax"]])
#             cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

#             label_text = ""
#             if DRAW_TRACK_ID:
#                 label_text += f"ID {tid}"
#             if DRAW_TEXT and isinstance(bbox_row["actions"], str) and bbox_row["actions"].strip():
#                 label_text += f": {bbox_row['actions']}"

#             if label_text:
#                 (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
#                 overlay = frame.copy()
#                 cv2.rectangle(overlay, (x1, max(y1 - th - 6, 0)), (x1 + tw + 4, y1), color, -1)
#                 cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)
#                 cv2.putText(frame, label_text, (x1 + 2, max(y1 - 4, 15)),
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)

#         writer.write(frame)
#         frame_idx += 1

#     cap.release()
#     writer.release()
#     print(f"✅ Saved: {out_path} ({frame_idx} frames)")

# print(f"\n🎬 Done! All {NUM_VIDEOS} bbox visualizations saved in: {OUTPUT_DIR}")


#### Split the Dataset 

In [36]:
# import os
# import pandas as pd
# from sklearn.model_selection import train_test_split

# # ============================================================
# # CONFIGURATION
# # ============================================================
# DATASET_DIR = "/home1/jtt_1/mtp/timesformer-dataset-train"
# CSV_PATH = os.path.join(DATASET_DIR, "labels.csv")

# TRAIN_SPLIT = 0.8
# SEED = 42

# # ============================================================
# # LOAD DATASET CSV
# # ============================================================
# if not os.path.exists(CSV_PATH):
#     raise FileNotFoundError(f"❌ CSV not found: {CSV_PATH}")

# df = pd.read_csv(CSV_PATH)
# print(f"📄 Loaded {len(df)} videos from {CSV_PATH}")

# # Optional: check columns
# expected_cols = {"Drone", "TimeOfDay", "VideoID", "VideoPath", "BBoxCSVPath"}
# missing_cols = expected_cols - set(df.columns)
# if missing_cols:
#     print(f"⚠️ Missing expected columns: {missing_cols}")

# # ============================================================
# # TRAIN / VAL SPLIT
# # ============================================================
# train_df, val_df = train_test_split(
#     df,
#     test_size=(1 - TRAIN_SPLIT),
#     random_state=SEED,
#     shuffle=True,
#     stratify=df["TimeOfDay"] if "TimeOfDay" in df.columns else None
# )

# print(f"🟢 Training videos: {len(train_df)}")
# print(f"🔵 Validation videos: {len(val_df)}")

# # ============================================================
# # SAVE NEW CSV FILES
# # ============================================================
# train_csv = os.path.join(DATASET_DIR, "train_labels.csv")
# val_csv = os.path.join(DATASET_DIR, "val_labels.csv")

# train_df.to_csv(train_csv, index=False)
# val_df.to_csv(val_csv, index=False)

# # ============================================================
# # SUMMARY CHECK
# # ============================================================
# print(f"\n✅ Train/Val split complete!")
# print(f"📁 Train CSV: {train_csv} ({len(train_df)} entries)")
# print(f"📁 Val CSV:   {val_csv} ({len(val_df)} entries)")

# # Distribution summary
# if "TimeOfDay" in df.columns:
#     print("\n📊 Distribution by TimeOfDay:")
#     print("Train:\n", train_df["TimeOfDay"].value_counts())
#     print("Val:\n", val_df["TimeOfDay"].value_counts())


#### Model Train Scene level script

In [37]:
# import os
# import torch
# from torch.utils.data import Dataset, DataLoader
# from transformers import TimesformerForVideoClassification, AutoImageProcessor, AdamW, get_scheduler
# from decord import VideoReader, cpu
# from sklearn.preprocessing import MultiLabelBinarizer
# from sklearn.metrics import f1_score
# import pandas as pd
# import numpy as np
# from tqdm import tqdm

# # ============================================================
# # CONFIGURATION
# # ============================================================
# BASE_DIR = "/home1/jtt_1/mtp/timesformer-dataset-train"
# TRAIN_CSV = os.path.join(BASE_DIR, "train_labels.csv")
# VAL_CSV = os.path.join(BASE_DIR, "val_labels.csv")
# MODEL_PATH = "/home1/jtt_1/mtp/timesformer_base_patch16_224_k400"
# SAVE_DIR = os.path.join("/home1/jtt_1/mtp", "trained-timesformer-model")
# os.makedirs(SAVE_DIR, exist_ok=True)

# NUM_FRAMES = 4
# EPOCHS = 10
# BATCH_SIZE = 1
# LR = 1e-4
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"🚀 Training on device: {DEVICE}")

# # ============================================================
# # DATASET CLASS
# # ============================================================
# class VideoDataset(Dataset):
#     def __init__(self, csv_file, processor, label_encoder, num_frames=8):
#         self.df = pd.read_csv(csv_file)
#         self.processor = processor
#         self.label_encoder = label_encoder
#         self.num_frames = num_frames

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]

#         # ---- FIX: Handle duplicate or relative paths safely ----
#         video_path = str(row["VideoPath"]).strip()
#         if not os.path.isabs(video_path):
#             # Remove any leading redundant prefix like 'timesformer-dataset-train/'
#             if "timesformer-dataset-train" in video_path:
#                 video_path = video_path.split("timesformer-dataset-train/")[-1]
#             video_path = os.path.join(BASE_DIR, video_path)
#         video_path = os.path.normpath(video_path)

#         # Skip missing videos
#         if not os.path.exists(video_path):
#             print(f"⚠️ Missing video: {video_path}, skipping...")
#             dummy = torch.zeros((3, self.num_frames, 224, 224))
#             labels = torch.zeros(len(self.label_encoder.classes_))
#             return dummy, labels

#         # ---- Parse labels ----
#         actions = str(row["Actions"]).replace("[", "").replace("]", "").replace("'", "").split(",")
#         actions = [a.strip() for a in actions if a.strip()]
#         labels = self.label_encoder.transform([actions])[0]

#         # ---- Load video frames ----
#         try:
#             vr = VideoReader(video_path, ctx=cpu(0))
#             total_frames = len(vr)
#             indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)
#             frames = [vr[i].asnumpy() for i in indices]
#         except Exception as e:
#             print(f"⚠️ Error reading {video_path}: {e}")
#             dummy = torch.zeros((3, self.num_frames, 224, 224))
#             labels = torch.zeros(len(self.label_encoder.classes_))
#             return dummy, labels

#         # ---- Preprocess ----
#         inputs = self.processor(frames, return_tensors="pt")
#         pixel_values = inputs["pixel_values"].squeeze(0)
#         return pixel_values, torch.tensor(labels, dtype=torch.float32)

# # ============================================================
# # LOAD LABELS & ENCODER
# # ============================================================
# train_df = pd.read_csv(TRAIN_CSV)
# val_df = pd.read_csv(VAL_CSV)

# # Unique actions
# all_actions = sorted(list(set(
#     sum([str(a).replace("[", "").replace("]", "").replace("'", "").split(",") for a in train_df["Actions"]], [])
# )))
# label_encoder = MultiLabelBinarizer()
# label_encoder.fit([all_actions])
# num_classes = len(all_actions)
# print(f"🎯 {num_classes} unique actions detected: {all_actions}")

# # ============================================================
# # MODEL SETUP
# # ============================================================
# processor = AutoImageProcessor.from_pretrained(MODEL_PATH)
# model = TimesformerForVideoClassification.from_pretrained(MODEL_PATH)

# # Replace classification head
# model.classifier = torch.nn.Linear(model.classifier.in_features, num_classes)
# model.to(DEVICE)

# # ============================================================
# # DATALOADERS
# # ============================================================
# train_ds = VideoDataset(TRAIN_CSV, processor, label_encoder, NUM_FRAMES)
# val_ds = VideoDataset(VAL_CSV, processor, label_encoder, NUM_FRAMES)
# train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
# val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# # ============================================================
# # OPTIMIZER / LOSS / SCHEDULER
# # ============================================================
# optimizer = AdamW(model.parameters(), lr=LR)
# num_training_steps = EPOCHS * len(train_loader)
# lr_scheduler = get_scheduler("linear", optimizer=optimizer,
#                              num_warmup_steps=0, num_training_steps=num_training_steps)
# criterion = torch.nn.BCEWithLogitsLoss()
# scaler = torch.amp.GradScaler('cuda')

# best_val_f1 = 0.0
# log_file = os.path.join(SAVE_DIR, "training_log.txt")

# # ============================================================
# # TRAINING LOOP
# # ============================================================
# for epoch in range(EPOCHS):
#     model.train()
#     train_loss = 0.0
#     for pixel_values, labels in tqdm(train_loader, desc=f"🟢 Epoch {epoch+1}/{EPOCHS} [Train]"):
#         pixel_values = pixel_values.to(DEVICE, non_blocking=True)
#         labels = labels.to(DEVICE, non_blocking=True)

#         with torch.amp.autocast('cuda'):
#             outputs = model(pixel_values)
#             loss = criterion(outputs.logits, labels)

#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         optimizer.zero_grad(set_to_none=True)
#         lr_scheduler.step()
#         train_loss += loss.item()

#     avg_train_loss = train_loss / len(train_loader)

#     # ---------------- VALIDATION ----------------
#     model.eval()
#     val_loss = 0.0
#     preds_all, targets_all = [], []

#     with torch.no_grad():
#         for pixel_values, labels in tqdm(val_loader, desc=f"🔵 Epoch {epoch+1}/{EPOCHS} [Val]"):
#             pixel_values = pixel_values.to(DEVICE)
#             labels = labels.to(DEVICE)

#             outputs = model(pixel_values)
#             loss = criterion(outputs.logits, labels)
#             val_loss += loss.item()

#             preds = (torch.sigmoid(outputs.logits) > 0.5).int().cpu().numpy()
#             preds_all.extend(preds)
#             targets_all.extend(labels.cpu().numpy())

#     avg_val_loss = val_loss / len(val_loader)
#     val_f1 = f1_score(targets_all, preds_all, average="micro")

#     print(f"\n📊 Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | F1: {val_f1:.4f}")

#     # Save best model
#     if val_f1 > best_val_f1:
#         best_val_f1 = val_f1
#         best_path = os.path.join(SAVE_DIR, f"best_timesformer_epoch{epoch+1}.pt")
#         torch.save(model.state_dict(), best_path)
#         print(f"✅ Saved new best model → {best_path} (F1={val_f1:.4f})")

#     with open(log_file, "a") as f:
#         f.write(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}, val_f1={val_f1:.4f}\n")

# print(f"\n🏁 Training complete! Best F1: {best_val_f1:.4f}")
# print(f"🧾 Logs → {log_file}")
# print(f"💾 Models saved in → {SAVE_DIR}")


### Person level Dataset creation

#### Train Data conversion

In [38]:
# import os
# import cv2
# import pandas as pd
# import numpy as np
# from tqdm import tqdm

# # ============================================================
# # CONFIGURATION
# # ============================================================
# ROOT = "/home1/jtt_1/mtp/timesformer-dataset-train"
# OUTPUT_ROOT = "/home1/jtt_1/mtp/timesformer-person-dataset"

# FRAME_SIZE = (224, 224)  # resize for TimeSformer
# CLIP_LEN = 16             # number of frames per clip
# FPS = 10
# SAVE_PREVIEWS = True
# SPLITS = ["train", "val"]  # process both

# os.makedirs(OUTPUT_ROOT, exist_ok=True)

# # ============================================================
# # FUNCTION: Extract Clips from Video + BBoxes
# # ============================================================
# def extract_person_clips(video_path, bbox_csv, video_id, split, preview_dir):
#     if not os.path.exists(video_path):
#         print(f"⚠️ Missing video file: {video_path}")
#         return []

#     if not os.path.exists(bbox_csv):
#         print(f"⚠️ Missing bbox CSV: {bbox_csv}")
#         return []

#     df_bbox = pd.read_csv(bbox_csv)
#     df_bbox = df_bbox[(df_bbox["lost"] == 0) & (df_bbox["occluded"] == 0)]

#     if df_bbox.empty:
#         print(f"⚠️ No valid bounding boxes for {video_id}")
#         return []

#     cap = cv2.VideoCapture(video_path)
#     total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#     num_tracks = len(df_bbox["track_id"].unique())
#     print(f"🎥 {split.upper()} → {video_id} | Persons: {num_tracks} | Frames: {total_frames}")

#     output_dir = os.path.join(OUTPUT_ROOT, split)
#     os.makedirs(output_dir, exist_ok=True)

#     results = []
#     total_clip_count = 0

#     for track_id in df_bbox["track_id"].unique():
#         track_df = df_bbox[df_bbox["track_id"] == track_id].sort_values("frame")
#         frames_list = track_df["frame"].unique()

#         if len(frames_list) < CLIP_LEN:
#             print(f"  ⚠️ Track {track_id}: too short ({len(frames_list)} frames), skipping.")
#             continue

#         clips_for_track = 0
#         preview_saved = False

#         for start_idx in range(0, len(frames_list) - CLIP_LEN + 1, CLIP_LEN):
#             clip_frames = frames_list[start_idx:start_idx + CLIP_LEN]
#             clip_actions = set()

#             # collect all action labels in this clip
#             for _, row in track_df[track_df["frame"].isin(clip_frames)].iterrows():
#                 acts = str(row["actions"]).split(",")
#                 clip_actions.update([a.strip() for a in acts if a.strip()])

#             if not clip_actions:
#                 continue

#             start_f, end_f = clip_frames[0], clip_frames[-1]
#             out_file = f"{video_id}_tid{track_id}_clip{start_f:04d}.mp4"
#             output_path = os.path.join(output_dir, out_file)

#             if os.path.exists(output_path):
#                 continue

#             writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, FRAME_SIZE)

#             for fnum in clip_frames:
#                 if fnum >= total_frames:
#                     continue
#                 cap.set(cv2.CAP_PROP_POS_FRAMES, fnum)
#                 ret, frame = cap.read()
#                 if not ret:
#                     continue

#                 row = track_df[track_df["frame"] == fnum].iloc[0]
#                 x1, y1, x2, y2 = map(int, [row["xmin"], row["ymin"], row["xmax"], row["ymax"]])
#                 person_crop = frame[y1:y2, x1:x2]
#                 if person_crop.size == 0:
#                     continue

#                 person_crop = cv2.resize(person_crop, FRAME_SIZE)
#                 writer.write(person_crop)

#                 # save preview once per track
#                 if SAVE_PREVIEWS and not preview_saved:
#                     preview_frame = frame.copy()
#                     cv2.rectangle(preview_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
#                     cv2.putText(preview_frame, f"{video_id} TID {track_id}", (x1, y1 - 10),
#                                 cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
#                     preview_path = os.path.join(preview_dir, f"{video_id}_tid{track_id}.jpg")
#                     cv2.imwrite(preview_path, preview_frame)
#                     preview_saved = True

#             writer.release()

#             if os.path.exists(output_path):
#                 results.append({
#                     "VideoID": video_id,
#                     "TrackID": track_id,
#                     "VideoPath": output_path,
#                     "Actions": ",".join(sorted(list(clip_actions))),
#                     "StartFrame": int(start_f),
#                     "EndFrame": int(end_f),
#                     "FrameCount": len(clip_frames)
#                 })
#                 clips_for_track += 1
#                 total_clip_count += 1

#         # print(f"  🧍 Track {track_id}: {clips_for_track} clips saved.")

#     cap.release()
#     print(f"✅ {video_id} done → {total_clip_count} total clips.\n")
#     return results, total_clip_count


# # ============================================================
# # MAIN LOOP FOR BOTH TRAIN AND VAL
# # ============================================================
# overall_stats = {}

# for split in SPLITS:
#     labels_csv = os.path.join(ROOT, f"{split}_labels.csv")
#     if not os.path.exists(labels_csv):
#         print(f"⚠️ Missing file for {split}: {labels_csv}")
#         continue

#     preview_dir = os.path.join(OUTPUT_ROOT, f"{split}_previews")
#     if SAVE_PREVIEWS:
#         os.makedirs(preview_dir, exist_ok=True)

#     df = pd.read_csv(labels_csv)
#     print(f"\n🚀 Starting {split.upper()} split | {len(df)} videos\n")

#     person_rows = []
#     total_clips = 0

#     for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Extracting {split} person clips"):
#         video_path = str(row["VideoPath"]).strip()
#         bbox_csv = str(row["BBoxCSVPath"]).strip()
#         video_id = str(row["VideoID"]).strip()

#         if not (os.path.exists(video_path) and os.path.exists(bbox_csv)):
#             print(f"⚠️ Missing data for {video_id}")
#             continue

#         clips, clip_count = extract_person_clips(video_path, bbox_csv, video_id, split, preview_dir)
#         person_rows.extend(clips)
#         total_clips += clip_count

#     # ---- Save split CSV ----
#     out_csv = os.path.join(OUTPUT_ROOT, f"{split}_person_labels.csv")
#     pd.DataFrame(person_rows).to_csv(out_csv, index=False)

#     overall_stats[split] = {
#         "videos": len(df),
#         "clips": total_clips,
#         "csv": out_csv,
#         "preview_dir": preview_dir
#     }

# # ============================================================
# # FINAL SUMMARY
# # ============================================================
# print("\n🏁 ALL SPLITS COMPLETE!\n")
# for split, info in overall_stats.items():
#     print(f"📂 {split.upper()} SPLIT SUMMARY:")
#     print(f"  🎞️ Videos processed: {info['videos']}")
#     print(f"  🎬 Person clips: {info['clips']}")
#     print(f"  🧾 Labels CSV: {info['csv']}")
#     print(f"  🖼️ Previews: {info['preview_dir']}\n")


In [39]:
TRAIN_CSV = "/home1/jtt_1/mtp/timesformer-person-dataset/train_person_labels.csv"
VAL_CSV   = "/home1/jtt_1/mtp/timesformer-person-dataset/val_person_labels.csv"


#### Test Datset Conversion

In [40]:
# import os
# import cv2
# import pandas as pd
# import numpy as np
# from tqdm import tqdm

# # ============================================================
# # CONFIGURATION
# # ============================================================
# ROOT = "/home1/jtt_1/mtp/timesformer-dataset-test"
# OUTPUT_ROOT = "/home1/jtt_1/mtp/timesformer-person-dataset-test"

# FRAME_SIZE = (224, 224)  # resize for TimeSformer
# CLIP_LEN = 16             # number of frames per clip
# FPS = 10
# SAVE_PREVIEWS = True
# SPLITS = ["test"]  # only process test split

# os.makedirs(OUTPUT_ROOT, exist_ok=True)

# # ============================================================
# # FUNCTION: Extract Clips from Video + BBoxes
# # ============================================================
# def extract_person_clips(video_path, bbox_csv, video_id, split, preview_dir):
#     if not os.path.exists(video_path):
#         print(f"⚠️ Missing video file: {video_path}")
#         return []

#     if not os.path.exists(bbox_csv):
#         print(f"⚠️ Missing bbox CSV: {bbox_csv}")
#         return []

#     df_bbox = pd.read_csv(bbox_csv)
#     df_bbox = df_bbox[(df_bbox["lost"] == 0) & (df_bbox["occluded"] == 0)]

#     if df_bbox.empty:
#         print(f"⚠️ No valid bounding boxes for {video_id}")
#         return []

#     cap = cv2.VideoCapture(video_path)
#     total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#     num_tracks = len(df_bbox["track_id"].unique())
#     print(f"🎥 {split.upper()} → {video_id} | Persons: {num_tracks} | Frames: {total_frames}")

#     output_dir = os.path.join(OUTPUT_ROOT, split)
#     os.makedirs(output_dir, exist_ok=True)

#     results = []
#     total_clip_count = 0

#     for track_id in df_bbox["track_id"].unique():
#         track_df = df_bbox[df_bbox["track_id"] == track_id].sort_values("frame")
#         frames_list = track_df["frame"].unique()

#         if len(frames_list) < CLIP_LEN:
#             print(f"  ⚠️ Track {track_id}: too short ({len(frames_list)} frames), skipping.")
#             continue

#         clips_for_track = 0
#         preview_saved = False

#         for start_idx in range(0, len(frames_list) - CLIP_LEN + 1, CLIP_LEN):
#             clip_frames = frames_list[start_idx:start_idx + CLIP_LEN]
#             clip_actions = set()

#             for _, row in track_df[track_df["frame"].isin(clip_frames)].iterrows():
#                 acts = str(row["actions"]).split(",")
#                 clip_actions.update([a.strip() for a in acts if a.strip()])

#             if not clip_actions:
#                 continue

#             start_f, end_f = clip_frames[0], clip_frames[-1]
#             out_file = f"{video_id}_tid{track_id}_clip{start_f:04d}.mp4"
#             output_path = os.path.join(output_dir, out_file)

#             writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, FRAME_SIZE)

#             for fnum in clip_frames:
#                 if fnum >= total_frames:
#                     continue
#                 cap.set(cv2.CAP_PROP_POS_FRAMES, fnum)
#                 ret, frame = cap.read()
#                 if not ret:
#                     continue

#                 row = track_df[track_df["frame"] == fnum].iloc[0]
#                 x1, y1, x2, y2 = map(int, [row["xmin"], row["ymin"], row["xmax"], row["ymax"]])
#                 person_crop = frame[y1:y2, x1:x2]
#                 if person_crop.size == 0:
#                     continue

#                 person_crop = cv2.resize(person_crop, FRAME_SIZE)
#                 writer.write(person_crop)

#                 if SAVE_PREVIEWS and not preview_saved:
#                     preview_frame = frame.copy()
#                     cv2.rectangle(preview_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
#                     cv2.putText(preview_frame, f"{video_id} TID {track_id}", (x1, y1 - 10),
#                                 cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
#                     preview_path = os.path.join(preview_dir, f"{video_id}_tid{track_id}.jpg")
#                     cv2.imwrite(preview_path, preview_frame)
#                     preview_saved = True

#             writer.release()

#             if os.path.exists(output_path):
#                 results.append({
#                     "VideoID": video_id,
#                     "TrackID": track_id,
#                     "VideoPath": output_path,
#                     "Actions": ",".join(sorted(list(clip_actions))),
#                     "StartFrame": int(start_f),
#                     "EndFrame": int(end_f),
#                     "FrameCount": len(clip_frames)
#                 })
#                 clips_for_track += 1
#                 total_clip_count += 1

#     cap.release()
#     print(f"✅ {video_id} done → {total_clip_count} total clips.\n")
#     return results, total_clip_count

# # ============================================================
# # MAIN LOOP FOR TEST SPLIT
# # ============================================================
# overall_stats = {}

# for split in SPLITS:
#     # ✅ Auto-detect labels.csv or test_labels.csv
#     test_csv_path = os.path.join(ROOT, "test_labels.csv")
#     labels_csv = test_csv_path if os.path.exists(test_csv_path) else os.path.join(ROOT, "labels.csv")

#     if not os.path.exists(labels_csv):
#         print(f"❌ No label file found in {ROOT}")
#         continue

#     preview_dir = os.path.join(OUTPUT_ROOT, f"{split}_previews")
#     if SAVE_PREVIEWS:
#         os.makedirs(preview_dir, exist_ok=True)

#     df = pd.read_csv(labels_csv)
#     print(f"\n🚀 Starting {split.upper()} split | {len(df)} videos\n")

#     person_rows = []
#     total_clips = 0

#     for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Extracting {split} person clips"):
#         video_path = str(row["VideoPath"]).strip()
#         bbox_csv = str(row["BBoxCSVPath"]).strip()
#         video_id = str(row["VideoID"]).strip()

#         if not (os.path.exists(video_path) and os.path.exists(bbox_csv)):
#             print(f"⚠️ Missing data for {video_id}")
#             continue

#         clips, clip_count = extract_person_clips(video_path, bbox_csv, video_id, split, preview_dir)
#         person_rows.extend(clips)
#         total_clips += clip_count

#     out_csv = os.path.join(OUTPUT_ROOT, f"{split}_person_labels.csv")
#     pd.DataFrame(person_rows).to_csv(out_csv, index=False)

#     overall_stats[split] = {
#         "videos": len(df),
#         "clips": total_clips,
#         "csv": out_csv,
#         "preview_dir": preview_dir
#     }

# # ============================================================
# # FINAL SUMMARY
# # ============================================================
# print("\n🏁 ALL SPLITS COMPLETE!\n")
# for split, info in overall_stats.items():
#     print(f"📂 {split.upper()} SPLIT SUMMARY:")
#     print(f"  🎞️ Videos processed: {info['videos']}")
#     print(f"  🎬 Person clips: {info['clips']}")
#     print(f"  🧾 Labels CSV: {info['csv']}")
#     print(f"  🖼️ Previews: {info['preview_dir']}\n")


#### check_dataset_consistency

In [41]:
# import os
# import pandas as pd
# import numpy as np
# from collections import Counter
# from tabulate import tabulate

# # ============================================================
# # CONFIGURATION
# # ============================================================
# ROOT_DIR = "/home1/jtt_1/mtp"  # base directory for datasets
# DATASETS = {
#     "train": os.path.join(ROOT_DIR, "timesformer-person-dataset/train_person_labels.csv"),
#     "val": os.path.join(ROOT_DIR, "timesformer-person-dataset/val_person_labels.csv"),
#     "test": os.path.join(ROOT_DIR, "timesformer-person-dataset-test/test_person_labels.csv")
# }

# CHECK_VIDEOS = True  # verify that .mp4 files exist
# TOP_N = 10           # show top-N actions by frequency

# # ============================================================
# # HELPER FUNCTION
# # ============================================================
# def load_actions(df):
#     actions = []
#     for a in df["Actions"]:
#         if isinstance(a, str):
#             actions.extend([x.strip() for x in a.split(",") if x.strip()])
#     return actions

# # ============================================================
# # MAIN ANALYSIS
# # ============================================================
# print("\n🚀 Dataset Consistency Check: Okutama TimeSformer Person-Level\n")

# summary_stats = {}
# all_action_sets = {}

# for split, path in DATASETS.items():
#     if not os.path.exists(path):
#         print(f"⚠️ Missing file for {split.upper()}: {path}")
#         continue

#     df = pd.read_csv(path)
#     print(f"\n📂 {split.upper()} — Loaded {len(df)} person clips from {path}")

#     # ---- Check missing or invalid paths ----
#     if CHECK_VIDEOS:
#         missing_videos = [p for p in df["VideoPath"] if not os.path.exists(p)]
#         if missing_videos:
#             print(f"  ⚠️ Missing videos: {len(missing_videos)} (e.g., {missing_videos[:3]})")
#         else:
#             print(f"  ✅ All video files exist.")

#     # ---- Count actions ----
#     actions = load_actions(df)
#     action_counts = Counter(actions)
#     total_actions = sum(action_counts.values())
#     unique_actions = len(action_counts)

#     print(f"  🎯 Unique actions: {unique_actions}")
#     print(f"  🧩 Total action tags (across clips): {total_actions}")
#     print(f"  🔝 Top-{TOP_N} actions:")
#     print(tabulate(action_counts.most_common(TOP_N), headers=["Action", "Count"], tablefmt="github"))

#     summary_stats[split] = {
#         "clips": len(df),
#         "unique_actions": unique_actions,
#         "total_actions": total_actions
#     }
#     all_action_sets[split] = set(action_counts.keys())

# # ============================================================
# # CROSS-SPLIT CONSISTENCY
# # ============================================================
# print("\n🔍 Cross-Split Action Consistency\n")

# all_classes = sorted(set().union(*all_action_sets.values()))
# print(f"🧠 Total unique actions across all splits: {len(all_classes)}")
# print(f"📋 Actions: {all_classes}\n")

# # Compare presence across splits
# comparison_rows = []
# for a in all_classes:
#     row = [a]
#     for split in DATASETS.keys():
#         row.append("✅" if a in all_action_sets.get(split, set()) else "❌")
#     comparison_rows.append(row)

# print(tabulate(comparison_rows, headers=["Action", "Train", "Val", "Test"], tablefmt="grid"))

# # ============================================================
# # SUMMARY TABLE
# # ============================================================
# print("\n📊 Dataset Summary Overview\n")
# rows = []
# for split, stats in summary_stats.items():
#     rows.append([
#         split.upper(),
#         stats["clips"],
#         stats["unique_actions"],
#         stats["total_actions"]
#     ])

# print(tabulate(rows, headers=["Split", "Clips", "Unique Actions", "Total Action Tags"], tablefmt="github"))

# # ============================================================
# # DATASET BALANCE METRIC
# # ============================================================
# print("\n⚖️ Class Balance Overview (Train vs Val)\n")
# if "train" in all_action_sets and "val" in all_action_sets:
#     train_only = all_action_sets["train"] - all_action_sets["val"]
#     val_only = all_action_sets["val"] - all_action_sets["train"]

#     if train_only:
#         print(f"  ⚠️ Actions only in TRAIN: {train_only}")
#     if val_only:
#         print(f"  ⚠️ Actions only in VAL: {val_only}")
#     if not train_only and not val_only:
#         print("  ✅ Train and Val action sets match perfectly.")

# print("\n✅ Consistency check complete!\n")


#### Multi-Clip Grid Visualizer (Final Version)

In [42]:
# import os
# import cv2
# import pandas as pd
# import numpy as np
# import math
# from tqdm import tqdm
# import random

# # ============================================================
# # CONFIGURATION
# # ============================================================
# DATASET_ROOT = "/home1/jtt_1/mtp/timesformer-person-dataset-test"
# SPLIT = "test"
# CSV_PATH = os.path.join(DATASET_ROOT, f"{SPLIT}_person_labels.csv")

# # Automatically pick up to N random clips
# MAX_SAMPLES = 16
# CLIP_SIZE = (256, 256)  # per-clip resolution
# OUTPUT_PATH = os.path.join(DATASET_ROOT, f"{SPLIT}_grid_preview_random.mp4")
# FPS = 10

# # ============================================================
# # LOAD DATA
# # ============================================================
# if not os.path.exists(CSV_PATH):
#     raise FileNotFoundError(f"❌ Could not find: {CSV_PATH}")

# df = pd.read_csv(CSV_PATH)
# print(f"📄 Loaded {len(df)} person clips from {CSV_PATH}")

# if len(df) == 0:
#     raise ValueError("❌ CSV is empty — no person clips found!")

# # Randomly select subset
# df_sample = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=None).reset_index(drop=True)
# print(f"🎲 Randomly selected {len(df_sample)} clips for visualization\n")

# # ============================================================
# # DYNAMIC GRID SIZE CALCULATION
# # ============================================================
# n_clips = len(df_sample)
# grid_side = math.ceil(math.sqrt(n_clips))  # best square fit
# GRID_ROWS, GRID_COLS = grid_side, grid_side
# GRID_SIZE = GRID_ROWS * GRID_COLS
# print(f"🧩 Auto Grid Layout: {GRID_ROWS} × {GRID_COLS} (for {n_clips} clips)")

# # ============================================================
# # LOAD CLIPS
# # ============================================================
# clips = []
# for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="📥 Loading clips"):
#     clip_path = str(row["VideoPath"]).strip()
#     if not os.path.isabs(clip_path):
#         clip_path = os.path.join(DATASET_ROOT, clip_path)
#     clip_path = os.path.normpath(clip_path)

#     if not os.path.exists(clip_path):
#         print(f"⚠️ Missing clip: {clip_path}")
#         continue

#     cap = cv2.VideoCapture(clip_path)
#     frames = []
#     while True:
#         ret, frame = cap.read()
#         if not ret:
#             break
#         frame = cv2.resize(frame, CLIP_SIZE)
#         frames.append(frame)
#     cap.release()

#     if len(frames) == 0:
#         continue

#     clips.append({
#         "frames": frames,
#         "video_id": row.get("VideoID", "unknown"),
#         "track_id": row.get("TrackID", "N/A"),
#         "actions": str(row["Actions"])
#     })

# if len(clips) == 0:
#     raise RuntimeError("❌ No valid clips loaded. Check dataset paths.")

# print(f"✅ Loaded {len(clips)} valid clips for grid preview.")

# # ============================================================
# # GRID COMPOSITION SETUP
# # ============================================================
# grid_width = CLIP_SIZE[0] * GRID_COLS
# grid_height = CLIP_SIZE[1] * GRID_ROWS
# max_frames = max(len(c["frames"]) for c in clips)

# out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (grid_width, grid_height))
# print(f"\n🎬 Generating random grid preview → {OUTPUT_PATH}")
# print(f"🧩 Layout: {GRID_ROWS}×{GRID_COLS} | Resolution: {grid_width}×{grid_height}\n")

# # ============================================================
# # FRAME-BY-FRAME GRID COMPOSITION
# # ============================================================
# for f in tqdm(range(max_frames), desc="🎞️ Composing grid"):
#     grid_cells = []
#     clip_idx = 0

#     for r in range(GRID_ROWS):
#         row_frames = []
#         for c in range(GRID_COLS):
#             if clip_idx < len(clips):
#                 clip = clips[clip_idx]
#                 frames = clip["frames"]
#                 frame = frames[min(f, len(frames)-1)]  # freeze last frame if shorter
#                 display = frame.copy()

#                 # Overlay info box
#                 y_offset = 25
#                 cv2.rectangle(display, (0, 0), (CLIP_SIZE[0], y_offset + 25), (0, 0, 0), -1)
#                 cv2.putText(display, f"{clip['video_id']} | TID {clip['track_id']}",
#                             (5, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)
#                 cv2.putText(display, f"{clip['actions']}",
#                             (5, y_offset + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
#             else:
#                 display = np.zeros((CLIP_SIZE[1], CLIP_SIZE[0], 3), dtype=np.uint8)

#             row_frames.append(display)
#             clip_idx += 1

#         grid_row = np.hstack(row_frames)
#         grid_cells.append(grid_row)

#     grid_frame = np.vstack(grid_cells)
#     out.write(grid_frame)

# out.release()
# print(f"\n✅ Saved random grid preview video → {OUTPUT_PATH}")
# print("👀 Open the video to verify crops, actions, and diversity visually.\n")


#### Training for person-level

In [43]:
# alreeady created py file

#### testing for person-level

In [1]:
# ============================================================
# evaluate_person_timesformer.py
# ============================================================
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from decord import VideoReader, cpu
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_recall_fscore_support, average_precision_score
from torch.utils.data import Dataset, DataLoader
from transformers import TimesformerForVideoClassification, AutoImageProcessor

# ============================================================
# CONFIGURATION
# ============================================================
BASE_DIR = "/home1/jtt_1/mtp/timesformer-person-dataset-test"
TEST_CSV = os.path.join(BASE_DIR, "test_person_labels.csv")

MODEL_PATH = "/home1/jtt_1/mtp/trained-timesformer-person/best_timesformer_epoch68.pt"
PRETRAIN_PATH = "/home1/jtt_1/mtp/timesformer_base_patch16_224_k400"

SAVE_DIR = "/home1/jtt_1/mtp/timesformer-eval-results"
os.makedirs(SAVE_DIR, exist_ok=True)

NUM_FRAMES = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
THRESH = 0.5

print(f"🚀 Evaluating on {DEVICE}")

# ============================================================
# DATASET CLASS
# ============================================================
class PersonVideoDataset(Dataset):
    def __init__(self, csv_file, processor, label_encoder, num_frames=8):
        self.df = pd.read_csv(csv_file)
        self.processor = processor
        self.label_encoder = label_encoder
        self.num_frames = num_frames

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_path = str(row["VideoPath"]).strip()
        if not os.path.isabs(video_path):
            video_path = os.path.join(BASE_DIR, video_path)
        video_path = os.path.normpath(video_path)

        actions = [a.strip() for a in str(row["Actions"]).split(",") if a.strip()]
        labels = self.label_encoder.transform([actions])[0]

        try:
            vr = VideoReader(video_path, ctx=cpu(0))
            total_frames = len(vr)
            indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)
            frames = [vr[i].asnumpy() for i in indices]
            inputs = self.processor(frames, return_tensors="pt")
            pixel_values = inputs["pixel_values"].squeeze(0)
        except Exception as e:
            print(f"⚠️ Error reading {video_path}: {e}")
            pixel_values = torch.zeros((3, self.num_frames, 224, 224))

        return pixel_values, torch.tensor(labels, dtype=torch.float32)

# ============================================================
# LOAD LABELS + MODEL
# ============================================================
test_df = pd.read_csv(TEST_CSV)

def extract_actions(df):
    all_actions = []
    for a in df["Actions"]:
        actions = [x.strip() for x in str(a).split(",") if x.strip()]
        all_actions.extend(actions)
    return sorted(list(set(all_actions)))

actions_list = extract_actions(test_df)
mlb = MultiLabelBinarizer()
mlb.fit([actions_list])
num_classes = len(actions_list)

print(f"🎯 Found {num_classes} unique actions: {actions_list}")

processor = AutoImageProcessor.from_pretrained(PRETRAIN_PATH)
model = TimesformerForVideoClassification.from_pretrained(PRETRAIN_PATH)
model.classifier = torch.nn.Linear(model.classifier.in_features, num_classes)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

# ============================================================
# DATA LOADER
# ============================================================
test_ds = PersonVideoDataset(TEST_CSV, processor, mlb, NUM_FRAMES)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2)

# ============================================================
# INFERENCE LOOP
# ============================================================
all_preds, all_targets, all_probs = [], [], []

print(f"\n🎞️ Running inference on {len(test_ds)} person clips...\n")
for pixel_values, labels in tqdm(test_loader):
    pixel_values = pixel_values.to(DEVICE)
    labels = labels.to(DEVICE)

    with torch.no_grad():
        outputs = model(pixel_values)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()
        preds = (probs > THRESH).astype(int)

    all_preds.extend(preds)
    all_probs.extend(probs)
    all_targets.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_targets = np.array(all_targets)

# ============================================================
# METRICS
# ============================================================
print("\n📊 Evaluating performance metrics...\n")

prec, rec, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average="micro", zero_division=0)
macro_prec, macro_rec, macro_f1, _ = precision_recall_fscore_support(all_targets, all_preds, average="macro", zero_division=0)
mAP = average_precision_score(all_targets, all_probs, average="macro")

print(f"✅ Micro Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
print(f"✅ Macro Precision: {macro_prec:.4f} | Recall: {macro_rec:.4f} | F1: {macro_f1:.4f}")
print(f"✅ Mean Average Precision (mAP): {mAP:.4f}")

# ============================================================
# PER-CLASS REPORT
# ============================================================
per_class_prec, per_class_rec, per_class_f1, _ = precision_recall_fscore_support(
    all_targets, all_preds, average=None, zero_division=0
)
per_class_ap = average_precision_score(all_targets, all_probs, average=None)

report = pd.DataFrame({
    "Action": actions_list,
    "Precision": per_class_prec,
    "Recall": per_class_rec,
    "F1": per_class_f1,
    "AP": per_class_ap
})

csv_path = os.path.join(SAVE_DIR, "test_class_report.csv")
report.to_csv(csv_path, index=False)

print(f"\n📁 Per-class results saved to: {csv_path}")
print(f"💾 Results directory: {SAVE_DIR}")

print("\n🏁 Evaluation complete!")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


🚀 Evaluating on cuda
🎯 Found 13 unique actions: ['Calling', 'Carrying', 'Drinking', 'Handshaking', 'Hugging', 'Lying', 'Pushing or Pulling', 'Reading', 'Running', 'Sitting', 'Standing', 'Walking', 'nan']

🎞️ Running inference on 4250 person clips...



100%|██████████| 4250/4250 [07:54<00:00,  8.95it/s]


📊 Evaluating performance metrics...

✅ Micro Precision: 0.5879 | Recall: 0.5202 | F1: 0.5520
✅ Macro Precision: 0.4376 | Recall: 0.2956 | F1: 0.3259
✅ Mean Average Precision (mAP): 0.3457

📁 Per-class results saved to: /home1/jtt_1/mtp/timesformer-eval-results/test_class_report.csv
💾 Results directory: /home1/jtt_1/mtp/timesformer-eval-results

🏁 Evaluation complete!


#### Visualization

In [8]:
# ============================================================
# visualize_predictions_grid.py
# ============================================================
import os
import cv2
import pandas as pd
import numpy as np
import math
from tqdm import tqdm
import torch
from transformers import TimesformerForVideoClassification, AutoImageProcessor
from decord import VideoReader, cpu

# ============================================================
# CONFIGURATION
# ============================================================
DATASET_ROOT = "/home1/jtt_1/mtp/timesformer-person-dataset-test"
CSV_PATH = os.path.join(DATASET_ROOT, "test_person_labels.csv")
MODEL_PATH = "/home1/jtt_1/mtp/models-2/best_timesformer_epoch68.pt"
PRETRAIN_PATH = "/home1/jtt_1/mtp/timesformer_base_patch16_224_k400"

# Random selection + video output
MAX_SAMPLES = 9
CLIP_SIZE = (256, 256)
FPS = 10
OUTPUT_PATH = os.path.join(DATASET_ROOT, "test_pred_gt_comparison.mp4")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_FRAMES = 8
THRESH = 0.4

print(f"🚀 Using device: {DEVICE}")

# ============================================================
# LOAD MODEL
# ============================================================
processor = AutoImageProcessor.from_pretrained(PRETRAIN_PATH)
model = TimesformerForVideoClassification.from_pretrained(PRETRAIN_PATH)
# We don’t know number of classes yet — we’ll rebuild head dynamically after reading dataset

# ============================================================
# LOAD DATA
# ============================================================
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"❌ Missing test CSV: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
print(f"📄 Loaded {len(df)} clips from {CSV_PATH}")

if len(df) == 0:
    raise ValueError("❌ CSV is empty!")

# Get unique action labels for correct head size
def extract_actions(df):
    all_actions = []
    for a in df["Actions"]:
        acts = [x.strip() for x in str(a).split(",") if x.strip()]
        all_actions.extend(acts)
    return sorted(list(set(all_actions)))

all_actions = extract_actions(df)
num_classes = len(all_actions)
print(f"🎯 Detected {num_classes} unique actions: {all_actions}")

# Rebuild classification head
model.classifier = torch.nn.Linear(model.classifier.in_features, num_classes)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

# Random sample
df_sample = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=None).reset_index(drop=True)
print(f"🎲 Selected {len(df_sample)} random clips for visualization\n")

# ============================================================
# GRID CONFIG
# ============================================================
n_clips = len(df_sample)
grid_side = math.ceil(math.sqrt(n_clips))
GRID_ROWS, GRID_COLS = grid_side, grid_side
grid_width = CLIP_SIZE[0] * GRID_COLS
grid_height = CLIP_SIZE[1] * GRID_ROWS
print(f"🧩 Grid Layout: {GRID_ROWS}×{GRID_COLS}")

out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (grid_width, grid_height))

# ============================================================
# PROCESS EACH CLIP
# ============================================================
clips = []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="🎥 Preparing clips"):
    clip_path = str(row["VideoPath"]).strip()
    if not os.path.isabs(clip_path):
        clip_path = os.path.join(DATASET_ROOT, clip_path)
    clip_path = os.path.normpath(clip_path)

    if not os.path.exists(clip_path):
        print(f"⚠️ Missing: {clip_path}")
        continue

    gt_actions = [a.strip() for a in str(row["Actions"]).split(",") if a.strip()]

    # Load clip frames
    vr = VideoReader(clip_path, ctx=cpu(0))
    total_frames = len(vr)
    indices = np.linspace(0, total_frames - 1, NUM_FRAMES, dtype=int)
    frames = [vr[i].asnumpy() for i in indices]

    # Predict
    inputs = processor(frames, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]
        pred_idx = np.where(probs > THRESH)[0]
        pred_actions = [all_actions[i] for i in pred_idx]

    # Match colors
    correct = set(gt_actions).intersection(set(pred_actions))
    incorrect = set(pred_actions) - correct
    missed = set(gt_actions) - correct

    cap = cv2.VideoCapture(clip_path)
    frames_cv = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, CLIP_SIZE)
        frames_cv.append(frame)
    cap.release()

    clips.append({
        "frames": frames_cv,
        "video_id": row.get("VideoID", "unknown"),
        "track_id": row.get("TrackID", "N/A"),
        "gt": gt_actions,
        "pred": pred_actions,
        "correct": list(correct),
        "incorrect": list(incorrect),
        "missed": list(missed)
    })

if len(clips) == 0:
    raise RuntimeError("❌ No valid clips loaded!")

# ============================================================
# COMPOSE GRID VIDEO
# ============================================================
max_frames = max(len(c["frames"]) for c in clips)
print(f"🎞️ Generating {OUTPUT_PATH} with {max_frames} frames")

for f in tqdm(range(max_frames), desc="🧩 Composing grid"):
    grid_cells = []
    clip_idx = 0
    for r in range(GRID_ROWS):
        row_frames = []
        for c in range(GRID_COLS):
            if clip_idx < len(clips):
                clip = clips[clip_idx]
                frames = clip["frames"]
                frame = frames[min(f, len(frames)-1)].copy()

                # Info overlay
                cv2.rectangle(frame, (0, 0), (CLIP_SIZE[0], 55), (0, 0, 0), -1)
                y = 18

                # Header
                cv2.putText(frame, f"{clip['video_id']} | TID {clip['track_id']}", (5, y),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)

                # GT in green
                gt_text = "GT: " + ", ".join(clip['gt'])
                cv2.putText(frame, gt_text, (5, y + 17),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

                # Prediction color-coded
                pred_text = "PRED: " + ", ".join(clip['pred'])
                color = (0, 255, 0) if set(clip["pred"]) == set(clip["gt"]) else (0, 0, 255)
                cv2.putText(frame, pred_text, (5, y + 34),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            else:
                frame = np.zeros((CLIP_SIZE[1], CLIP_SIZE[0], 3), dtype=np.uint8)
            row_frames.append(frame)
            clip_idx += 1
        grid_row = np.hstack(row_frames)
        grid_cells.append(grid_row)

    grid_frame = np.vstack(grid_cells)
    out.write(grid_frame)

out.release()
print(f"\n✅ Saved visualization → {OUTPUT_PATH}")
print("🟩 Green = Correct, 🟥 Red = Wrong/Missing Prediction")


🚀 Using device: cuda
📄 Loaded 4250 clips from /home1/jtt_1/mtp/timesformer-person-dataset-test/test_person_labels.csv
🎯 Detected 13 unique actions: ['Calling', 'Carrying', 'Drinking', 'Handshaking', 'Hugging', 'Lying', 'Pushing or Pulling', 'Reading', 'Running', 'Sitting', 'Standing', 'Walking', 'nan']
🎲 Selected 9 random clips for visualization

🧩 Grid Layout: 3×3


🎥 Preparing clips: 100%|██████████| 9/9 [00:01<00:00,  6.15it/s]


🎞️ Generating /home1/jtt_1/mtp/timesformer-person-dataset-test/test_pred_gt_comparison.mp4 with 16 frames


🧩 Composing grid: 100%|██████████| 16/16 [00:00<00:00, 395.92it/s]


✅ Saved visualization → /home1/jtt_1/mtp/timesformer-person-dataset-test/test_pred_gt_comparison.mp4
🟩 Green = Correct, 🟥 Red = Wrong/Missing Prediction


In [ ]:
# df -su